# Explainer Video Pipeline — Kaggle (free GPU)

**Before running:** open the right sidebar →
1. **Settings → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → On** (required for installs, git, model download, Groq/Pixabay)

Then Run All. The finished video is saved to **/kaggle/working/explainer_videos** (Output tab, downloadable).


## 0. GPU check + clone

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU! Right sidebar -> Settings -> Accelerator -> GPU T4 x2 (or P100), then Run All."
)
print("GPU:", torch.cuda.get_device_name(0))

import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"            # model cache (persists in session)
os.environ["VIBEVOICE_MODEL"] = "1.5B"                        # 0.5B / 1.5B / Large
os.environ["VIDEO_OUTPUT_DIR"] = "/kaggle/working/explainer_videos"  # final video lands here

%cd /kaggle/working
!rm -rf audio && git clone https://github.com/saitejatiru/audiowithvideoremotion.git audio
%cd audio
!rm -rf VibeVoice && git clone https://github.com/vibevoice-community/VibeVoice.git
import sys; sys.path.insert(0, ".")

## 1. Install dependencies (~5-8 min first run)

In [ ]:
!pip install -q fastapi "uvicorn[standard]" soundfile requests nemo-text-processing gradio tenacity "pandas<3.0.0" "huggingface-hub>=1.2.0"
!pip install -q -r align/requirements.txt
!pip install -q openai json-repair
!pip install -q -e VibeVoice

# Manim — real 2D animation (CPU render)
!apt-get -qq install -y libcairo2-dev libpango1.0-dev ffmpeg >/dev/null 2>&1
!pip install -q manim

# Node.js + Remotion for rendering
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1
!apt-get install -y nodejs >/dev/null 2>&1
!cd video && npm install
!apt-get install -yq libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpangocairo-1.0-0 libgtk-3-0 fonts-liberation >/dev/null 2>&1
print("Deps installed.")

## 2. API keys — paste yours, both lines must print OK

In [ ]:
import os

# LLM: GROQ (free, reliable). Key from console.groq.com
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["LLM_API_KEY"]  = "<PASTE_YOUR_GROQ_KEY>"      # gsk_...
os.environ["LLM_MODEL"]    = "llama-3.3-70b-versatile"

# Visuals: Pixabay (free). Key from pixabay.com/api/docs
os.environ["PIXABAY_API_KEY"] = "<PASTE_YOUR_PIXABAY_KEY>"

os.environ["GRADIO_SHARE"] = "1"   # Kaggle exposes the UI via Gradio's share link

# smoke test
if "<" in os.environ["LLM_API_KEY"]:
    print("!! LLM_API_KEY placeholder — paste your Groq key.")
else:
    from openai import OpenAI
    _c = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
    try:
        _r = _c.chat.completions.create(model=os.environ["LLM_MODEL"],
            messages=[{"role":"user","content":"Reply with the word: ok"}], max_tokens=10)
        print("LLM OK:", os.environ["LLM_MODEL"], "->", _r.choices[0].message.content)
    except Exception as e:
        print("LLM BROKEN:", repr(e))

if "<" in os.environ["PIXABAY_API_KEY"]:
    print("!! PIXABAY_API_KEY placeholder — paste your Pixabay key.")
else:
    import requests
    try:
        _n = requests.get("https://pixabay.com/api/", params={"key": os.environ["PIXABAY_API_KEY"],
            "q":"human heart","image_type":"illustration","per_page":3}, timeout=10).json().get("totalHits",0)
        print("PIXABAY OK — illustrations available:", _n)
    except Exception as e:
        print("PIXABAY BROKEN:", repr(e))

## 3. Launch the UI

Kaggle has no localhost proxy, so this uses Gradio's **public share link** (`https://xxxx.gradio.live`).
Click it, paste a script, Generate. The finished video is saved to **/kaggle/working/explainer_videos**
(right sidebar → Output → download). If the share link fails to appear, re-run this cell.

In [ ]:
%cd /kaggle/working/audio
!python app.py